# Predictive-maintenance model selection under missing sensor data and shared repair capacity

**Experiment notebook** for the manuscript
*"Selecting Predictive-Maintenance Models Under Missing Sensor Data and Shared Repair Capacity"*

**Experimental design**

* Missingness settings: **MCAR, MAR, MNAR, MIXED**, each at **10 %, 30 %, 50 %, 70 %, 90 %** total missingness of the
  predictor matrix X (20 settings).
* **5 seeds**: training masks 0–4, validation masks 1000–1004, test masks 2000–2004.
* Masks touch **only the predictors X** (retained sensor channels). `unit`, `cycle`, the RUL target and everything derived
  from it are never masked; assertions check that y has no missing values.
* **Exact realised rates**: every mask has exactly `round(rate·n·p)` missing cells (intercept calibrated on the realised count).
* **MIXED** = equal-parts protocol: one third MCAR, one third MAR, remainder MNAR; disjoint components (component code saved).
* No imputation pipeline and no mask-derived predictor feature is used. XGBoost receives the NaN-containing
  predictors directly (native missing-value handling).
* One cached mask per data set × split × mechanism × rate × seed, shared by every compared strategy
  (`masks_v5/`); target and realised rates are written to `mask_rates.csv` separately for train / validation / test.

What the notebook does (Runtime ▸ Run all):

1. Downloads NASA C-MAPSS (no manual download is needed).
2. Generates and caches the masks.
3. Trains three families on identical incomplete data: **prediction-only** (native XGBoost; complete-data reference),
   **regret-calibrated** (safety margin chosen on validation regret), **cost-aware training** (quantile regression at the
   theoretical and at a validated quantile; hybrid squared-error + regret-surrogate loss).
4. Solves the capacity-constrained scheduling MILP (HiGHS via `scipy.optimize.milp`) for every prediction vector and for the
   perfect-information plan → realised cost, failures, downtime, **regret**.
5. Runs the shift setting (train MCAR-10 → test MNAR-30) and the cost / capacity sensitivity grids (on MNAR-30).
6. Writes summary CSVs, a JSON summary, and **all table bodies and figures of the manuscript** to `OUT_DIR/paper/`.

**Scientific-integrity notes**
* No number in the manuscript should be entered by hand: everything comes from `OUT_DIR/`.
* `SMOKE_TEST=True` runs the pipeline on a tiny *synthetic* stand-in **only to check that the code runs**; its tables and
  figures are watermarked and must never be reported.
* Seeds quantify within-setting variability; settings of one data set share the fleet and are not independent replications.

**Resumable:** results are appended to `results_raw.csv` after every block (`RESUME=True`); wall-clock time is accumulated
over sessions in `runtime_sessions.csv`.

## 0. Configuration
Edit only this cell. Everything below is driven by `CONFIG`.

In [ ]:
CONFIG = dict(
    # ---- data -------------------------------------------------------------
    DATASETS      = ["FD001","FD002","FD003","FD004"],
    DATA_DIR      = "./data/cmapss",    # where train_FD00x.txt etc. live (downloaded or manually uploaded)
    SMOKE_TEST    = False,              # True -> synthetic stand-in data, code check only, NEVER report results

    # ---- experimental matrix -------------------------------------------------
    # Seeds: training masks use `seed`, validation masks `seed + 1000`, test masks `seed + 2000`
    #   -> training 0-4, validation 1000-1004, test 2000-2004.
    SEEDS         = [0, 1, 2, 3, 4],
    VAL_SEED_OFFSET  = 1000,
    TEST_SEED_OFFSET = 2000,
    MECHANISMS    = ["MCAR", "MAR", "MNAR", "MIXED"],
    RATES         = [10, 30, 50,70,90],       # total missingness (%) of the predictor matrix X; realised rate is exact
    # SCENARIOS is derived below: "<MECHANISM>-<RATE>", e.g. "MIXED-30"
    SHIFT_SCENARIOS = [("MCAR-10", "MNAR-30")],   # (train/val setting, test setting)
    SENS_SCENARIO = "MNAR-30",          # setting used for the cost / capacity sensitivity grids
    SENS_N_SEEDS  = 2,                  # sensitivity grids use the first two seeds
    # Strategies. "X_margin" = predictions of X shifted by a safety margin chosen on validation regret (regret calibration).
    # All incomplete-data strategies feed NaN-containing predictors directly to XGBoost (native missing-value handling).
    # No imputation pipeline and no mask-derived predictor feature is used anywhere.
    METHODS       = ["xgb_native", "oracle_complete_xgb",
                     "xgb_native_margin", "oracle_complete_xgb_margin",
                     "xgb_quantile",            # quantile regression at the theoretical quantile c_e/(c_e+c_d) (Proposition 1)
                     "xgb_quantile_val",        # quantile regression with alpha chosen on validation regret
                     "dami"],                   # hybrid squared-error + regret-surrogate loss (lambda chosen on validation regret)
    QUANTILE_GRID = [0.02, 0.05, 0.10, 0.20, 0.35, 0.50],
    MARGIN_GRID   = [0, 5, 10, 15, 20, 25, 30, 35, 40, 50, 60],   # safety margins (cycles) tried for the regret-calibrated variants
    VAL_TRUNCATE  = True,      # truncate validation trajectories at random cycles so that validation RUL > 0 (mimics the test files)

    # ---- missingness generator ---------------------------------------------
    MNAR_STRENGTH    = 2.0,    # slope of logistic(|z|): own-value dependence
    MAR_STRENGTH     = 2.0,    # slope of logistic(z_anchor): dependence on the always-observed anchor channel
    MASK_VERSION     = "v5",   # masks are cached under masks_<version>/

    # ---- features / target --------------------------------------------------
    WINDOW        = 20,
    RUL_CLIP      = 125,
    TRAIN_STRIDE  = 2,         # use every k-th training cycle (speed); 1 = all rows

    # ---- maintenance decision model (costs relative to c_pm = 1) -------------
    COSTS         = dict(c_pm=1.0, c_f=10.0, c_e=0.02, c_d=0.20),
    CAPACITY      = 2,          # units per period
    COST_GRID_CF  = [3.0, 5.0, 10.0, 20.0],
    COST_GRID_CE  = [0.02, 0.10, 0.20],
    CAP_GRID      = [1, 2, 3, 5],
    EARLY_PM_DELTA = 20,        # PM counted as "unnecessarily early" if it discards > delta cycles of life

    # ---- hybrid (DAMI) objective ----------------------------------------------
    LAMBDA_GRID   = [0.0, 5.0, 20.0, 50.0, 100.0, 500.0, 1000.0, 5000.0],
    TAU           = 3.0,         # sigmoid temperature (cycles) for the failure-penalty jump
    SELECT_BY     = "regret",    # "regret" (decision-aware selection) or "rmse" (ablation)

    # ---- xgboost -----------------------------------------------------------
    XGB_PARAMS    = dict(n_estimators=400, max_depth=5, learning_rate=0.05, subsample=0.8,
                         colsample_bytree=0.8, min_child_weight=5, reg_lambda=1.0, n_jobs=-1),

    OUT_DIR       = "./OUT_DIR",        # output folder: results, masks, summaries, paper tables and figures
    USE_DRIVE     = True,               # Colab: mount Google Drive and keep data/masks/results under MyDrive/OUT_DIR
    RESUME        = True,               # skip (dataset, scenario, seed, tag, cost, capacity) blocks already present in results_raw.csv
)
CONFIG["SCENARIOS"] = [f"{m}-{r}" for m in CONFIG["MECHANISMS"] for r in CONFIG["RATES"]]
assert CONFIG["SENS_SCENARIO"] in CONFIG["SCENARIOS"]
assert all(a in CONFIG["SCENARIOS"] and b in CONFIG["SCENARIOS"] for a, b in CONFIG["SHIFT_SCENARIOS"])

## 1. Environment

In [ ]:
import sys, subprocess, importlib

def _pip(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

for mod, pkg in [("xgboost", "xgboost"), ("sklearn", "scikit-learn"), ("scipy", "scipy>=1.9"),
                 ("pandas", "pandas"), ("matplotlib", "matplotlib"), ("requests", "requests")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        _pip([pkg])

import os, io, re, json, time, zipfile, warnings, platform
from pathlib import Path
import numpy as np, pandas as pd
import scipy, sklearn, xgboost as xgb
from scipy.optimize import milp, LinearConstraint, Bounds
from scipy import stats
from scipy import sparse
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and CONFIG["USE_DRIVE"]:
    from google.colab import drive
    drive.mount("/content/drive")
    CONFIG["OUT_DIR"]  = "/content/drive/MyDrive/OUT_DIR"
    CONFIG["DATA_DIR"] = "/content/drive/MyDrive/OUT_DIR/data/cmapss"
    print("Google Drive mounted; outputs persist in", CONFIG["OUT_DIR"])
OUT = Path(CONFIG["OUT_DIR"]); OUT.mkdir(parents=True, exist_ok=True)
(OUT / "figures").mkdir(exist_ok=True)

ENV = dict(python=platform.python_version(), numpy=np.__version__, pandas=pd.__version__, scipy=scipy.__version__,
           sklearn=sklearn.__version__, xgboost=xgb.__version__, colab=IN_COLAB,
           smoke_test=CONFIG["SMOKE_TEST"], timestamp=time.strftime("%Y-%m-%d %H:%M:%S"))
print(json.dumps(ENV, indent=2))
assert tuple(int(x) for x in scipy.__version__.split(".")[:2]) >= (1, 9), "scipy>=1.9 needed for scipy.optimize.milp (HiGHS)"

Mounted at /content/drive
Google Drive mounted; outputs persist in /content/drive/MyDrive/dami_outputs_v5
{
  "python": "3.13.15",
  "numpy": "2.1.3",
  "pandas": "2.2.3",
  "scipy": "1.16.3",
  "sklearn": "1.6.1",
  "xgboost": "3.4.1",
  "colab": true,
  "smoke_test": false,
  "timestamp": "2026-09-23 19:41:53"
}


## 2. Data acquisition
C-MAPSS is fetched automatically from the NASA PCoE repository (with fallbacks); no manual download is needed. If every
download fails, place `train_FD00x.txt`, `test_FD00x.txt`, `RUL_FD00x.txt` (x = 1–4) into `CONFIG["DATA_DIR"]` — in Colab
the cell will offer an upload dialog.

In [ ]:
CMAPSS_URLS = [
    # Official NASA PCoE mirror (S3). The zip contains a nested CMAPSSData.zip.
    "https://phm-datasets.s3.amazonaws.com/NASA/6.+Turbofan+Engine+Degradation+Simulation+Data+Set.zip",
    # Older data.nasa.gov endpoint (may redirect or be retired)
    "https://data.nasa.gov/download/ff5v-kuh6/application%2Fzip",
]

SENSOR_COLS = [f"s{i}" for i in range(1, 22)]
CMAPSS_COLS = ["unit", "cycle", "op1", "op2", "op3"] + SENSOR_COLS
# Sensors that are constant in FD001/FD003 (standard practice to drop them)
CONSTANT_SENSORS_FD001 = ["s1", "s5", "s6", "s10", "s16", "s18", "s19"]

def _extract_all_txt(zbytes, dest):
    """Recursively extract .txt files from a (possibly nested) zip."""
    with zipfile.ZipFile(io.BytesIO(zbytes)) as z:
        for name in z.namelist():
            if name.lower().endswith(".zip"):
                _extract_all_txt(z.read(name), dest)
            elif name.lower().endswith(".txt") and not name.endswith("/"):
                Path(dest, Path(name).name).write_bytes(z.read(name))

def cmapss_files_present(d, subsets):
    return all(Path(d, f"{p}_{s}.txt").exists() for s in subsets for p in ["train", "test", "RUL"])

def download_cmapss(d, subsets):
    d = Path(d); d.mkdir(parents=True, exist_ok=True)
    if cmapss_files_present(d, subsets):
        print("C-MAPSS files already present in", d); return True
    import requests
    for url in CMAPSS_URLS:
        try:
            print("Trying", url)
            r = requests.get(url, timeout=180); r.raise_for_status()
            _extract_all_txt(r.content, d)
            if cmapss_files_present(d, subsets):
                print("Downloaded C-MAPSS to", d); return True
        except Exception as e:
            print("  failed:", repr(e)[:200])
    if IN_COLAB:
        print("Automatic download failed. Please upload train_/test_/RUL_FD00x.txt files now.")
        from google.colab import files
        up = files.upload()
        for name, content in up.items():
            Path(d, Path(name).name).write_bytes(content)
    return cmapss_files_present(d, subsets)

def make_synthetic_cmapss(d, n_train=40, n_test=20, seed=0):
    """Tiny synthetic stand-in used ONLY for smoke tests of the code path (labelled as such everywhere)."""
    rng = np.random.default_rng(seed)
    d = Path(d); d.mkdir(parents=True, exist_ok=True)
    def traj(unit, n):
        t = np.arange(1, n + 1)
        deg = (t / n) ** 2
        S = np.column_stack([500 + 30 * deg * (1 + 0.3 * k) + rng.normal(0, 1.0, n) for k in range(21)])
        S[:, [0, 4, 5, 9, 15, 17, 18]] = 100.0  # constant channels
        op = np.zeros((n, 3))
        return np.column_stack([np.full(n, unit), t, op, S])
    tr = np.vstack([traj(u, rng.integers(150, 300)) for u in range(1, n_train + 1)])
    tests, ruls = [], []
    for u in range(1, n_test + 1):
        n = rng.integers(150, 300); cut = rng.integers(60, n - 10)
        tests.append(traj(u, n)[:cut]); ruls.append(n - cut)
    for s in CONFIG["DATASETS"]:
        np.savetxt(d / f"train_{s}.txt", tr, fmt="%.4f")
        np.savetxt(d / f"test_{s}.txt", np.vstack(tests), fmt="%.4f")
        np.savetxt(d / f"RUL_{s}.txt", np.array(ruls), fmt="%d")

if CONFIG["SMOKE_TEST"]:
    CONFIG["DATA_DIR"] = "./data/SYNTHETIC_SMOKE_TEST"
    make_synthetic_cmapss(CONFIG["DATA_DIR"])
    print("!!! SMOKE TEST: synthetic data in use. Do NOT report these results. !!!")
else:
    ok = download_cmapss(CONFIG["DATA_DIR"], CONFIG["DATASETS"])
    assert ok, ("C-MAPSS files not found. Download 'Turbofan Engine Degradation Simulation Data Set' from the NASA "
                "PCoE repository and place train_/test_/RUL_FD00x.txt in " + CONFIG["DATA_DIR"])

C-MAPSS files already present in /content/drive/MyDrive/dami_outputs_v5/data/cmapss


In [ ]:
def load_cmapss(subset, d):
    d = Path(d)
    tr = pd.read_csv(d / f"train_{subset}.txt", sep=r"\s+", header=None).iloc[:, :26]; tr.columns = CMAPSS_COLS
    te = pd.read_csv(d / f"test_{subset}.txt", sep=r"\s+", header=None).iloc[:, :26]; te.columns = CMAPSS_COLS
    rul = pd.read_csv(d / f"RUL_{subset}.txt", sep=r"\s+", header=None).iloc[:, 0].values.astype(float)
    tr["unit"] = tr["unit"].astype(int); te["unit"] = te["unit"].astype(int)
    tr["cycle"] = tr["cycle"].astype(int); te["cycle"] = te["cycle"].astype(int)
    maxc = tr.groupby("unit")["cycle"].transform("max")
    tr["RUL"] = (maxc - tr["cycle"]).astype(float)
    # test: true RUL at last cycle = RUL file; earlier cycles get RUL_last + (last - cycle)
    last = te.groupby("unit")["cycle"].transform("max")
    rul_map = dict(zip(sorted(te["unit"].unique()), rul))
    te["RUL"] = te["unit"].map(rul_map) + (last - te["cycle"])
    # drop channels constant in training
    const = [c for c in SENSOR_COLS if tr[c].std() < 1e-8]
    sensors = [c for c in SENSOR_COLS if c not in const]
    return tr, te, sensors

DATA = {}
for s in CONFIG["DATASETS"]:
    tr, te, sensors = load_cmapss(s, CONFIG["DATA_DIR"])
    DATA[s] = dict(train=tr, test=te, sensors=sensors)
    print(f"{s}: train rows={len(tr)}, units={tr.unit.nunique()}, test rows={len(te)}, units={te.unit.nunique()}, "
          f"sensors used={len(sensors)} {sensors}")

FD001: train rows=20631, units=100, test rows=13096, units=100, sensors used=15 ['s2', 's3', 's4', 's6', 's7', 's8', 's9', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']
FD002: train rows=53759, units=260, test rows=33991, units=259, sensors used=21 ['s1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21']
FD003: train rows=24720, units=100, test rows=16596, units=100, sensors used=16 ['s2', 's3', 's4', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's17', 's20', 's21']
FD004: train rows=61249, units=249, test rows=41214, units=248, sensors used=21 ['s1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10', 's11', 's12', 's13', 's14', 's15', 's16', 's17', 's18', 's19', 's20', 's21']


## 3. Missingness generators (predictors only, exact realised rates)
All generators operate on the **complete predictor matrix** of one split (rows = cycles, columns = retained sensor channels)
and return a boolean mask `M` (`True` = observed) plus a component code `C` (0 observed, 1 MCAR, 2 MAR, 3 MNAR).
The number of missing cells is exactly `k = round(rate·n·p)`, computed in integer arithmetic with ties rounded up (`n_missing_target`).

* **MCAR-r**: `k` cells drawn uniformly without replacement.
* **MAR-r**: missingness of every non-anchor channel follows `sigmoid(a + 2·z_anchor)`, where `z_anchor` is the standardised
  value of the first retained sensor channel, which is never masked.
* **MNAR-r**: missingness of a cell follows `sigmoid(a + 2·|z|)`, `z` = the cell's own standardised deviation from the healthy
  (first 30 cycles, training units) baseline.
* **MIXED-r**: `k//3` cells MCAR, `k//3` cells MAR, the remainder MNAR; each component is drawn only among cells that are still
  observed, so the components are disjoint; the anchor channel stays observed so that the MAR component remains MAR.

*Exact calibration.* A Bernoulli draw `u < sigmoid(a + s)` is equivalent to `logit(u) − s < a`. We draw `u` once and mask the
`k` eligible cells with the smallest `logit(u) − s`, i.e. the intercept `a` is set from the realised count instead of the
expected rate. Masks are cached in `masks_v5/` and logged in `mask_rates.csv`.

In [ ]:
# ---------------------------------------------------------------------------------------------
# Missingness is applied ONLY to the predictor matrix X (the retained sensor channels).
# unit, cycle and the RUL target (and anything derived from it) are never touched.
# Every generator returns an exact number of missing cells k = round(rate * n * p).
# ---------------------------------------------------------------------------------------------
def n_missing_target(rate_pct, n_cells):
    """k = round(rate * n * p) in exact integer arithmetic (ties rounded up). Floating-point products such as
    0.1 * 249165 = 24916.5 are rounded inconsistently by round(); this helper is the single source of truth."""
    return (int(rate_pct) * int(n_cells) + 50) // 100

def _logit(u):
    u = np.clip(u, 1e-12, 1 - 1e-12); return np.log(u) - np.log1p(-u)

def _select_exact(score, eligible, k, rng):
    """Exact-count logistic masking.
    A Bernoulli draw with probability sigmoid(a + score) is `u < sigmoid(a + score)`  <=>  logit(u) - score < a.
    Instead of fixing the intercept a from the *expected* rate, we draw u once and take the k eligible cells with the
    smallest t = logit(u) - score, i.e. the intercept is calibrated on the *realised* count. score = 0 gives MCAR."""
    idx = np.flatnonzero(eligible.ravel())
    if k > len(idx): raise ValueError(f"cannot mask {k} cells: only {len(idx)} eligible")
    if k == 0: return np.zeros(eligible.size, bool).reshape(eligible.shape)
    t = _logit(rng.random(len(idx))) - score.ravel()[idx]
    chosen = idx[np.argpartition(t, k - 1)[:k]]
    out = np.zeros(eligible.size, bool); out[chosen] = True
    return out.reshape(eligible.shape)

def _scores(X, baseline_stats, cfg):
    """MAR score: depends only on the anchor channel (column 0), which is never masked under MAR / MIXED.
    MNAR score: depends on the cell's own absolute standardised deviation from the healthy training baseline."""
    Z = (X - baseline_stats["mean"]) / (baseline_stats["std"] + 1e-9)
    s_mar = np.repeat((cfg["MAR_STRENGTH"] * np.clip(Z[:, [0]], -5, 5)), X.shape[1], axis=1)
    s_mnar = cfg["MNAR_STRENGTH"] * np.clip(np.abs(Z), 0, 5)
    return s_mar, s_mnar

ANCHOR = 0   # first retained sensor channel; always observed under MAR and MIXED so that MAR stays MAR
COMPONENT_CODE = dict(OBS=0, MCAR=1, MAR=2, MNAR=3)

def gen_mask(X, scenario, seed, baseline_stats, cfg=CONFIG):
    """X: float array (n_rows x n_predictors) of COMPLETE predictor values. Returns (M, C, info):
    M bool, True = observed; C uint8 component code of every missing cell (0 observed, 1 MCAR, 2 MAR, 3 MNAR)."""
    X = np.asarray(X, float); n, p = X.shape
    assert not np.isnan(X).any(), "generator expects the complete predictor matrix"
    if scenario == "NONE":
        return np.ones((n, p), bool), np.zeros((n, p), np.uint8), dict(target_rate=0.0, k_total=0)
    mech, rate_pct = scenario.split("-"); rate = int(rate_pct) / 100.0
    rng = np.random.default_rng(seed)
    k = n_missing_target(rate_pct, n * p)
    s_mar, s_mnar = _scores(X, baseline_stats, cfg)
    all_cells = np.ones((n, p), bool)
    no_anchor = all_cells.copy(); no_anchor[:, ANCHOR] = False
    C = np.zeros((n, p), np.uint8)
    if mech == "MCAR":
        C[_select_exact(np.zeros((n, p)), all_cells, k, rng)] = COMPONENT_CODE["MCAR"]
    elif mech == "MAR":
        C[_select_exact(s_mar, no_anchor, k, rng)] = COMPONENT_CODE["MAR"]
    elif mech == "MNAR":
        C[_select_exact(s_mnar, all_cells, k, rng)] = COMPONENT_CODE["MNAR"]
    elif mech == "MIXED":
        # equal-parts protocol: total k cells; one third MCAR, one third MAR, remainder MNAR; components are disjoint
        # (each component is drawn only among cells not yet masked), so every missing cell has exactly one mechanism.
        k_mcar = k // 3; k_mar = k // 3; k_mnar = k - k_mcar - k_mar
        free = no_anchor.copy()                      # anchor stays observed so that the MAR component remains MAR
        sel = _select_exact(np.zeros((n, p)), free, k_mcar, rng); C[sel] = COMPONENT_CODE["MCAR"]; free &= ~sel
        sel = _select_exact(s_mar, free, k_mar, rng);             C[sel] = COMPONENT_CODE["MAR"];  free &= ~sel
        sel = _select_exact(s_mnar, free, k_mnar, rng);           C[sel] = COMPONENT_CODE["MNAR"]; free &= ~sel
    else:
        raise ValueError(scenario)
    M = C == 0
    n_miss = int((~M).sum())
    assert n_miss == k, (scenario, n_miss, k)                      # realised count is exact
    if mech in ("MAR", "MIXED"): assert M[:, ANCHOR].all()
    info = dict(target_rate=rate, k_total=k, n_cells=int(n * p), n_missing=n_miss, realised_rate=n_miss / (n * p),
                n_mcar=int((C == 1).sum()), n_mar=int((C == 2).sum()), n_mnar=int((C == 3).sum()))
    if mech == "MIXED":
        assert info["n_mcar"] == k // 3 and info["n_mar"] == k // 3 and info["n_mnar"] == k - 2 * (k // 3)
        assert info["n_mcar"] + info["n_mar"] + info["n_mnar"] == k
    return M, C, info

def healthy_baseline(train, sensors, n_early=30):
    early = train[train["cycle"] <= n_early]
    return dict(mean=early[sensors].mean().values, std=early[sensors].std().values)

MASK_DIR = OUT / f"masks_{CONFIG['MASK_VERSION']}"; MASK_DIR.mkdir(exist_ok=True)
MASK_LOG_CSV = OUT / "mask_rates.csv"
MASK_LOG = {}     # key -> row ; one row per dataset x split x scenario x mask seed
if CONFIG["RESUME"] and MASK_LOG_CSV.exists():
    for r_ in pd.read_csv(MASK_LOG_CSV).to_dict("records"):
        MASK_LOG[(r_["dataset"], r_["split"], r_["split_id"], r_["scenario"], int(r_["mask_seed"]))] = r_

def get_mask(dataset, split, split_id, df, sensors, scenario, mask_seed, baseline_stats):
    """One cached mask per dataset x split x mechanism x rate x seed. Every strategy of a block calls this function
    with identical arguments and therefore receives the identical mask (first call generates, later calls load)."""
    forbidden = {"RUL", "unit", "cycle"} & set(sensors)
    assert not forbidden, f"target / index columns must never be maskable: {forbidden}"
    f = MASK_DIR / f"{dataset}_{split_id}_{scenario}_s{mask_seed}.npz"
    X = df[sensors].values.astype(float)
    k_target = 0 if scenario == "NONE" else n_missing_target(scenario.split("-")[1], X.size)
    M = None
    if f.exists():
        z = np.load(f); M, C = z["M"], z["C"]
        assert M.shape == X.shape, f"cached mask {f.name} does not match the data"
        if int((~M).sum()) != k_target:      # mask cached by an earlier notebook revision with a different tie rounding
            print(f"  regenerating {f.name}: cached count {int((~M).sum())} != target {k_target}"); M = None
    if M is None:
        M, C, _ = gen_mask(X, scenario, mask_seed, baseline_stats)
        np.savez_compressed(f, M=M, C=C)
    n_cells = M.size; n_miss = int((~M).sum())
    target = 0.0 if scenario == "NONE" else int(scenario.split("-")[1]) / 100.0
    assert n_miss == k_target, (f.name, n_miss, k_target)
    MASK_LOG[(dataset, split, split_id, scenario, int(mask_seed))] = dict(
        dataset=dataset, split=split, split_id=split_id, scenario=scenario,
        mechanism=scenario.split("-")[0], mask_seed=int(mask_seed), n_rows=M.shape[0], n_predictors=M.shape[1],
        n_cells=n_cells, target_rate=target, n_missing=n_miss, realised_rate=n_miss / n_cells,
        n_mcar=int((C == 1).sum()), n_mar=int((C == 2).sum()), n_mnar=int((C == 3).sum()), mask_file=f.name)
    return M

def flush_mask_log():
    if MASK_LOG:
        pd.DataFrame(list(MASK_LOG.values())).sort_values(["dataset", "scenario", "split", "mask_seed"]).to_csv(MASK_LOG_CSV, index=False)

def apply_mask(df, sensors, M):
    """Return the NaN-containing predictor matrix. Only the predictor columns are touched."""
    X = df[sensors].values.astype(float).copy(); X[~M] = np.nan
    return X

# quick sanity check: realised rates are exact and MIXED components are disjoint thirds
_tr = DATA[CONFIG["DATASETS"][0]]["train"]; _s = DATA[CONFIG["DATASETS"][0]]["sensors"]
_bs = healthy_baseline(_tr, _s)
for sc in CONFIG["SCENARIOS"]:
    _M, _C, _i = gen_mask(_tr[_s].values, sc, 123, _bs)
    print(f"{sc:9s} target={_i['target_rate']:.2f} realised={_i['realised_rate']:.6f}  cells: MCAR={_i['n_mcar']} MAR={_i['n_mar']} MNAR={_i['n_mnar']}")

## 4. Feature construction (no imputation model, no mask-derived features)
For every (unit, cycle) and sensor: the current raw value (NaN if masked), the last observed value within the unit, and the
trailing-window mean and standard deviation over observed entries; plus the cycle count (an index, never masked).
Entries that cannot be computed remain NaN and are routed by XGBoost's native missing-value handling. No observed-fraction,
time-since-last-observation or missing-fraction feature is used.

In [ ]:
def build_features(df, sensors, M, window=CONFIG["WINDOW"]):
    """Features computed from the NaN-containing predictor matrix only. No imputation model and no mask-derived
    feature (observed fraction, time since last observation, missing fraction) is used: cells that cannot be
    computed stay NaN and are handled by XGBoost's native missing-value routing.
    Per sensor: current raw value (NaN if masked), last observed value within the unit (NaN before the first
    observation), trailing-window mean / std over the observed entries (NaN if too few). Plus the cycle count,
    which is an index variable, not a sensor, and is never masked."""
    X = apply_mask(df, sensors, M)
    out = {}
    g = pd.DataFrame(X, columns=sensors); g["unit"] = df["unit"].values
    grp = g.groupby("unit", sort=False)
    roll_mean = grp[sensors].rolling(window, min_periods=1).mean().reset_index(level=0, drop=True).sort_index()
    roll_std = grp[sensors].rolling(window, min_periods=2).std().reset_index(level=0, drop=True).sort_index()
    last_obs = grp[sensors].ffill()
    for j, s in enumerate(sensors):
        out[f"{s}_cur"] = X[:, j]
        out[f"{s}_last"] = last_obs[s].values
        out[f"{s}_mean{window}"] = roll_mean[s].values
        out[f"{s}_std{window}"] = roll_std[s].values
    out["cycle"] = df["cycle"].values.astype(float)
    F = pd.DataFrame(out)
    assert not any(c.startswith("RUL") for c in F.columns)
    return F

def last_cycle_rows(df):
    return df.groupby("unit")["cycle"].idxmax().values

## 5. Maintenance decision model and regret
Per-asset cost of serving at period $t$ when true residual life is $y$ (unit cost function of the paper):
$c(t;y)=c_{PM}+c_E(y-t)$ if $t\le y$, else $c_F+c_D(t-y)$.
Fleet plan: assign each unit to exactly one period, ≤ K units per period, minimise total cost (transportation-type
MILP, solved exactly with HiGHS). The planner uses predicted lives; the realised cost is computed with true lives.
Regret = realised cost − oracle (perfect-information) cost.

In [ ]:
def unit_cost_matrix(y, H, costs):
    """c[i,t] for t = 1..H (t index 0..H-1), y: array of residual lives (cycles)."""
    t = np.arange(1, H + 1)[None, :]
    yy = np.asarray(y, dtype=float)[:, None]
    early = costs["c_pm"] + costs["c_e"] * (yy - t)
    late = costs["c_f"] + costs["c_d"] * (t - yy)
    return np.where(t <= yy, early, late)

def solve_plan(y_plan, H, K, costs):
    """Solve the assignment MILP with planning lives y_plan. Returns period (1..H) chosen for each unit."""
    n = len(y_plan)
    C = unit_cost_matrix(y_plan, H, costs)
    c = C.ravel()
    # each unit exactly once
    var = np.arange(n * H)
    A_once = sparse.csr_matrix((np.ones(n * H), (var // H, var)), shape=(n, n * H))
    # capacity per period
    A_cap = sparse.csr_matrix((np.ones(n * H), (var % H, var)), shape=(H, n * H))
    cons = [LinearConstraint(A_once, 1, 1), LinearConstraint(A_cap, 0, K)]
    res = milp(c, constraints=cons, integrality=np.ones(n * H), bounds=Bounds(0, 1),
               options={"disp": False, "time_limit": 120})
    if res.x is None or res.status != 0:
        raise RuntimeError(f"MILP failed: {res.message}")
    x = res.x.reshape(n, H)
    return x.argmax(axis=1) + 1

def horizon_for(y_true, y_pred, K):
    n = len(y_true)
    H = int(np.ceil(max(np.max(y_true), np.max(y_pred)))) + 10
    return max(H, int(np.ceil(n / K)) + 1)

def evaluate_decision(y_true, y_pred, costs=CONFIG["COSTS"], K=CONFIG["CAPACITY"], early_delta=CONFIG["EARLY_PM_DELTA"]):
    """Plan with y_pred, evaluate with y_true. Returns dict of decision metrics."""
    y_true = np.asarray(y_true, float); y_pred = np.clip(np.round(np.asarray(y_pred, float)), 1, None)
    H = horizon_for(y_true, y_pred, K)
    t_hat = solve_plan(y_pred, H, K, costs)
    t_orc = solve_plan(y_true, H, K, costs)
    Ctrue = unit_cost_matrix(y_true, H, costs)
    idx = np.arange(len(y_true))
    cost_hat = Ctrue[idx, t_hat - 1].sum(); cost_orc = Ctrue[idx, t_orc - 1].sum()
    failures = int((t_hat > y_true).sum())
    downtime = float(np.clip(t_hat - y_true, 0, None).sum())
    early_pm = int(((y_true - t_hat) > early_delta).sum())
    # uncapacitated single-machine regret (closed form) for diagnostics
    r_unit = np.where(y_pred <= y_true, costs["c_e"] * (y_true - y_pred),
                      (costs["c_f"] - costs["c_pm"]) + costs["c_d"] * (y_pred - y_true)).sum()
    return dict(cost=float(cost_hat), oracle_cost=float(cost_orc), regret=float(cost_hat - cost_orc),
                norm_regret=float((cost_hat - cost_orc) / max(cost_orc, 1e-9)), failures=failures,
                downtime=downtime, early_pm=early_pm, unit_regret_uncap=float(r_unit), horizon=H, capacity=K)

# smoke check of the solver
_yt = np.array([10, 20, 30, 40.]); print(evaluate_decision(_yt, _yt + np.array([5, -5, 0, 15])))

{'cost': 26.1, 'oracle_cost': 4.0, 'regret': 22.1, 'norm_regret': 5.525, 'failures': 2, 'downtime': 20.0, 'early_pm': 0, 'unit_regret_uncap': 22.1, 'horizon': 65, 'capacity': 2}


## 6. Models
* `xgb_native`: squared-error XGBoost on the NaN-containing features (this is the hybrid loss with **λ = 0**).
* `oracle_complete_xgb`: the same model trained and tested without missing values (prediction reference).
* `*_margin`: predictions shifted down by a safety margin chosen on validation regret (regret calibration).
* `xgb_quantile`: quantile regression at α = c_e/(c_e+c_d) (Proposition 1, proportional case); `xgb_quantile_val`: α chosen on validation regret.
* `dami`: hybrid objective $(\hat y-y)^2 + \lambda\,\tilde r(\hat y;y)$, λ chosen from `LAMBDA_GRID` by validation regret.

All incomplete-data strategies of a block receive the identical three feature matrices (train / validation / test).

In [ ]:
def make_dami_objective(lam, costs, tau):
    c_e, c_d, J = costs["c_e"], costs["c_d"], costs["c_f"] - costs["c_pm"]
    def obj(y_true, y_pred):
        e = y_pred - y_true                    # >0: optimistic (late intervention)
        grad_sq = 2.0 * e
        hess_sq = 2.0 * np.ones_like(e)
        u = e / tau
        sig = 1.0 / (1.0 + np.exp(-np.clip(u, -30, 30)))
        dsig = sig * (1 - sig)
        grad_r = np.where(e < 0, -c_e, c_d) + (J / tau) * dsig
        hess_r = (J / tau ** 2) * dsig + 1e-6   # positive Gauss-Newton approximation
        return grad_sq + lam * grad_r, hess_sq + lam * hess_r
    return obj

def fit_xgb(Xtr, ytr, lam=0.0, costs=CONFIG["COSTS"], tau=CONFIG["TAU"], seed=0, params=CONFIG["XGB_PARAMS"]):
    """XGBoost on NaN-containing predictors: missing=np.nan -> default-direction (native) missing-value handling."""
    kw = dict(params); kw["random_state"] = seed
    assert not np.isnan(np.asarray(ytr, float)).any(), "the target must never contain missing values"
    if lam > 0:
        model = xgb.XGBRegressor(objective=make_dami_objective(lam, costs, tau), base_score=float(np.mean(ytr)), missing=np.nan, **kw)
    else:
        model = xgb.XGBRegressor(objective="reg:squarederror", base_score=float(np.mean(ytr)), missing=np.nan, **kw)
    model.fit(Xtr, ytr)
    return model

def rmse(a, b): return float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))
def mae(a, b): return float(np.mean(np.abs(np.asarray(a) - np.asarray(b))))
def phm08_score(y_true, y_pred):
    d = np.asarray(y_pred) - np.asarray(y_true)
    return float(np.sum(np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)))

## 7. Experiment runner (one dataset × scenario × seed)

In [ ]:
def split_train_val(train, seed, frac=0.2):
    units = np.sort(train["unit"].unique()); rng = np.random.default_rng(seed)
    val_units = rng.choice(units, size=max(1, int(frac * len(units))), replace=False)
    return train[~train["unit"].isin(val_units)].reset_index(drop=True), train[train["unit"].isin(val_units)].reset_index(drop=True)

def truncate_units(df, seed, lo=0.3, hi=0.95):
    """Truncate each unit's trajectory at a random fraction of its life (like the C-MAPSS test files)."""
    rng = np.random.default_rng(seed)
    parts = []
    for u, g in df.groupby("unit", sort=False):
        n = len(g); cut = int(np.clip(round(rng.uniform(lo, hi) * n), 5, n))
        parts.append(g.iloc[:cut])
    return pd.concat(parts).reset_index(drop=True)

def run_block(dataset, scen_train, scen_test, seed, cfg=CONFIG, methods=None, costs=None, K=None, tag="", existing=None):
    """Run the requested methods for one (dataset, train setting, test setting, seed).
    `existing` = set of method names already recorded for this block (resume)."""
    methods = list(methods or cfg["METHODS"]); costs = costs or cfg["COSTS"]; K = K or cfg["CAPACITY"]
    existing = set(existing or [])
    def need(m): return (m in methods) and (m not in existing)
    def need_family(base): return need(base) or need(base + "_margin")
    D = DATA[dataset]; sensors = D["sensors"]
    tr_all, te = D["train"], D["test"]
    tr, va = split_train_val(tr_all, seed)                       # split by unit
    if cfg.get("VAL_TRUNCATE", False):
        va = truncate_units(va, seed + 500)
    bs = healthy_baseline(tr, sensors)
    seed_tr, seed_va, seed_te = seed, seed + cfg["VAL_SEED_OFFSET"], seed + cfg["TEST_SEED_OFFSET"]
    # one cached mask per dataset x split x mechanism x rate x seed, shared by every strategy of the block
    M_tr = get_mask(dataset, "train", f"train_split{seed}", tr, sensors, scen_train, seed_tr, bs)
    M_va = get_mask(dataset, "val", f"val{'trunc' if cfg.get('VAL_TRUNCATE', False) else ''}_split{seed}", va, sensors, scen_train, seed_va, bs)
    M_te = get_mask(dataset, "test", "test", te, sensors, scen_test, seed_te, bs)
    flush_mask_log()
    # ---- targets: never masked ------------------------------------------------------------------
    ytr = np.minimum(tr["RUL"].values, cfg["RUL_CLIP"])
    te_idx = last_cycle_rows(te); yte = te["RUL"].values[te_idx]
    yva_last_idx = last_cycle_rows(va); yva_last = va["RUL"].values[yva_last_idx]
    for name_, y_ in [("train", tr["RUL"].values), ("train_clipped", ytr), ("val", va["RUL"].values), ("val_last", yva_last),
                      ("test", te["RUL"].values), ("test_last", yte)]:
        assert not np.isnan(np.asarray(y_, float)).any(), f"y ({name_}) contains missing values"
    rate_cols = dict(target_rate_train=(0.0 if scen_train == "NONE" else int(scen_train.split("-")[1]) / 100),
                     target_rate_val=(0.0 if scen_train == "NONE" else int(scen_train.split("-")[1]) / 100),
                     target_rate_test=(0.0 if scen_test == "NONE" else int(scen_test.split("-")[1]) / 100),
                     realised_rate_train=float(1 - M_tr.mean()), realised_rate_val=float(1 - M_va.mean()),
                     realised_rate_test=float(1 - M_te.mean()),
                     mask_seed_train=seed_tr, mask_seed_val=seed_va, mask_seed_test=seed_te,
                     n_val_units=int(len(yva_last)), n_test_units=int(len(yte)))
    val_oracle = evaluate_decision(yva_last, yva_last, costs=costs, K=K)      # capacity pressure of the validation fleet
    rate_cols.update(val_oracle_cost=val_oracle["oracle_cost"], val_oracle_failures=val_oracle["failures"])
    stride = np.arange(len(tr)) % cfg["TRAIN_STRIDE"] == 0
    rows = []; t0 = time.time()

    def base_row(method, yhat_te, yhat_va, lam=np.nan):
        dec = evaluate_decision(yte, yhat_te, costs=costs, K=K)
        return dict(dataset=dataset, scen_train=scen_train, scen_test=scen_test, seed=seed, method=method, tag=tag, lam=lam,
                    rmse=rmse(yte, yhat_te), mae=mae(yte, yhat_te), phm08=phm08_score(yte, yhat_te),
                    val_rmse=rmse(yva_last, yhat_va), c_f=costs["c_f"], c_e=costs["c_e"], c_d=costs["c_d"], **rate_cols, **dec)

    def record(method, yhat_te, yhat_va, extra=None, lam=np.nan):
        if method in existing: return
        row = base_row(method, yhat_te, yhat_va, lam)
        if extra: row.update(extra)
        rows.append(row)

    def record_with_margin(base, yhat_te, yhat_va, extra=None, lam=np.nan):
        """Record the base predictor and its regret-calibrated version (safety margin chosen on validation regret)."""
        record(base, yhat_te, yhat_va, extra, lam)
        if need(base + "_margin"):
            cand = [(evaluate_decision(yva_last, yhat_va - mg, costs=costs, K=K)["regret"], mg) for mg in cfg["MARGIN_GRID"]]
            vr, mg = min(cand)
            record(base + "_margin", yhat_te - mg, yhat_va - mg, dict(margin=mg, val_regret=vr, **(extra or {})), lam)

    # ---- native XGBoost on the NaN-containing predictors (the same three matrices feed every incomplete-data strategy)
    F_tr = build_features(tr, sensors, M_tr); F_va = build_features(va, sensors, M_va); F_te = build_features(te, sensors, M_te)

    def lambda_sweep(Ftr, Fva, Fte, name):
        cand = []
        for lam in cfg["LAMBDA_GRID"]:
            model = fit_xgb(Ftr[stride], ytr[stride], lam=lam, costs=costs, seed=seed)
            yv = model.predict(Fva.iloc[yva_last_idx]); yt = model.predict(Fte.iloc[te_idx])
            vr = evaluate_decision(yva_last, yv, costs=costs, K=K)["regret"]
            cand.append(dict(lam=lam, val_regret=vr, val_rmse=rmse(yva_last, yv), yt=yt, yv=yv))
            row = base_row(f"{name}_lam{lam:g}", yt, yv, lam); row["tag"] = tag + "|lambda_grid"; row["val_regret"] = vr
            rows.append(row)
        key = "val_regret" if cfg["SELECT_BY"] == "regret" else "val_rmse"
        best = min([c for c in cand if c["lam"] > 0], key=lambda c: c[key])
        return cand, best

    if need_family("xgb_native") or need("dami"):
        cand, best = lambda_sweep(F_tr, F_va, F_te, "dami")          # lambda = 0 is the native squared-error model
        lam0 = [c for c in cand if c["lam"] == 0][0]
        record_with_margin("xgb_native", lam0["yt"], lam0["yv"], lam=0.0)
        record("dami", best["yt"], best["yv"], dict(val_regret=best["val_regret"]), lam=best["lam"])

    # ---- quantile regression (Proposition 1) ---------------------------------------------------
    def fit_quantile(alpha):
        kw = dict(cfg["XGB_PARAMS"]); kw["random_state"] = seed
        return xgb.XGBRegressor(objective="reg:quantileerror", quantile_alpha=alpha, missing=np.nan, **kw).fit(F_tr[stride], ytr[stride])
    if need("xgb_quantile"):
        alpha = costs["c_e"] / (costs["c_e"] + costs["c_d"])
        qm = fit_quantile(alpha)
        record("xgb_quantile", qm.predict(F_te.iloc[te_idx]), qm.predict(F_va.iloc[yva_last_idx]), dict(quantile_alpha=alpha))
    if need("xgb_quantile_val"):
        cand = []
        for al in cfg["QUANTILE_GRID"]:
            qm = fit_quantile(al); yv = qm.predict(F_va.iloc[yva_last_idx])
            cand.append((evaluate_decision(yva_last, yv, costs=costs, K=K)["regret"], al, qm.predict(F_te.iloc[te_idx]), yv))
        vr, al, yt, yv = min(cand, key=lambda c: (c[0], c[1]))
        record("xgb_quantile_val", yt, yv, dict(quantile_alpha=al, val_regret=vr))

    # ---- complete-data prediction reference (no missing values; identical across settings by construction) -------
    if need_family("oracle_complete_xgb"):
        Fc_tr = build_features(tr, sensors, np.ones_like(M_tr, bool)); Fc_va = build_features(va, sensors, np.ones_like(M_va, bool))
        Fc_te = build_features(te, sensors, np.ones_like(M_te, bool))
        model = fit_xgb(Fc_tr[stride], ytr[stride], seed=seed)
        record_with_margin("oracle_complete_xgb", model.predict(Fc_te.iloc[te_idx]), model.predict(Fc_va.iloc[yva_last_idx]))

    # ---- perfect-information plan (decision oracle) -----------------------------
    if "oracle_perfect_info" not in existing:
        rows.append(base_row("oracle_perfect_info", yte, yva_last))
    print(f"  [{dataset} {scen_train}->{scen_test} seed={seed} {tag}] {len(rows)} new rows in {time.time() - t0:.0f}s")
    return rows

## 8. Run the main matrix (4 data sets × 20 settings × 5 seeds), the shift setting and the sensitivity grids

In [ ]:
RESULTS_CSV = OUT / "results_raw.csv"
ALL_ROWS = []
if CONFIG["RESUME"] and RESULTS_CSV.exists():
    _prev = pd.read_csv(RESULTS_CSV)
    if "smoke_test" in _prev and (_prev["smoke_test"].astype(bool) != CONFIG["SMOKE_TEST"]).any():
        raise RuntimeError("results_raw.csv mixes smoke-test and real runs. Delete it (or change OUT_DIR) before continuing.")
    _old = set(_prev["scen_test"].unique()) - set(CONFIG["SCENARIOS"])
    if _old or "realised_rate_test" not in _prev:
        raise RuntimeError(f"results_raw.csv comes from an earlier missing-data design {sorted(_old)}; use a fresh OUT_DIR.")
    ALL_ROWS = _prev.to_dict("records")
    print(f"RESUME: loaded {len(ALL_ROWS)} rows from {RESULTS_CSV}")

def block_key(dataset, scen_train, scen_test, seed, tag, costs, K):
    return (dataset, scen_train, scen_test, int(seed), tag, float(costs["c_f"]), float(costs["c_e"]), int(K))

def existing_methods(key):
    ds, st, se, seed, tag, cf, ce, K = key
    out = set()
    for r_ in ALL_ROWS:
        if (r_["dataset"], r_["scen_train"], r_["scen_test"], int(r_["seed"])) == (ds, st, se, seed) and \
           float(r_["c_f"]) == cf and float(r_["c_e"]) == ce and int(r_["capacity"]) == K and \
           str(r_["tag"]).split("|")[0] == tag:
            out.add(r_["method"])
    return out

def drop_rows(key, methods):
    global ALL_ROWS
    ds, st, se, seed, tag, cf, ce, K = key
    ALL_ROWS = [r_ for r_ in ALL_ROWS if not ((r_["dataset"], r_["scen_train"], r_["scen_test"], int(r_["seed"])) == (ds, st, se, seed)
                and float(r_["c_f"]) == cf and float(r_["c_e"]) == ce and int(r_["capacity"]) == K
                and str(r_["tag"]).split("|")[0] == tag and r_["method"] in methods)]

def flush():
    df = pd.DataFrame(ALL_ROWS)
    df["smoke_test"] = CONFIG["SMOKE_TEST"]
    tmp = RESULTS_CSV.with_suffix(".tmp.csv"); df.to_csv(tmp, index=False); tmp.replace(RESULTS_CSV)
    flush_mask_log()
    return df

def run_block_resumable(dataset, scen_train, scen_test, seed, methods=None, costs=None, K=None, tag=""):
    methods = list(methods or CONFIG["METHODS"]); costs = costs or CONFIG["COSTS"]; K = K or CONFIG["CAPACITY"]
    key = block_key(dataset, scen_train, scen_test, seed, tag, costs, K)
    have = existing_methods(key)
    # the hybrid-loss family must be complete for the current lambda grid; otherwise recompute the whole family
    sel = ["dami", "xgb_native", "xgb_native_margin"]
    if any(m in methods for m in sel):
        grid_rows = {f"dami_lam{l:g}" for l in CONFIG["LAMBDA_GRID"]}
        if have & grid_rows and not grid_rows <= have:
            stale = (have & grid_rows) | (set(sel) & have) | {m for m in have if m.startswith("dami_lam")}
            drop_rows(key, stale); have -= stale
            print(f"  lambda grid changed -> recomputing hybrid-loss family for {key[:5]}")
    missing = [m for m in methods if m not in have]
    if not missing and "oracle_perfect_info" in have:
        print(f"  skip (complete): {key[:5]}"); return []
    return run_block(dataset, scen_train, scen_test, seed, methods=methods, costs=costs, K=K, tag=tag, existing=have)

T_START = time.time()
# 8a. main matrix: same mechanism and rate in train/val/test  (4 data sets x 20 settings x 5 seeds = 400 blocks)
for ds in CONFIG["DATASETS"]:
    for sc in CONFIG["SCENARIOS"]:
        for seed in CONFIG["SEEDS"]:
            ALL_ROWS += run_block_resumable(ds, sc, sc, seed, tag="main"); flush()

# 8b. shift: train/val under one setting, test under another
for ds in CONFIG["DATASETS"]:
    for sc_tr, sc_te in CONFIG["SHIFT_SCENARIOS"]:
        for seed in CONFIG["SEEDS"]:
            ALL_ROWS += run_block_resumable(ds, sc_tr, sc_te, seed, tag="shift"); flush()

# 8c. sensitivity grids on SENS_SCENARIO: failure-cost ratio, early-replacement cost, capacity (first SENS_N_SEEDS seeds)
SENS_SCEN = CONFIG["SENS_SCENARIO"]
SENS_METHODS = [m for m in ["xgb_native", "xgb_native_margin", "xgb_quantile", "xgb_quantile_val", "dami"] if m in CONFIG["METHODS"]]
sens_seeds = CONFIG["SEEDS"][:CONFIG["SENS_N_SEEDS"]]
for ds in CONFIG["DATASETS"]:
    for cf in CONFIG["COST_GRID_CF"]:
        costs = dict(CONFIG["COSTS"]); costs["c_f"] = cf
        for seed in sens_seeds:
            ALL_ROWS += run_block_resumable(ds, SENS_SCEN, SENS_SCEN, seed, methods=SENS_METHODS, costs=costs, tag="sens_cf"); flush()
    for ce in CONFIG["COST_GRID_CE"]:
        if ce == CONFIG["COSTS"]["c_e"]: continue
        costs = dict(CONFIG["COSTS"]); costs["c_e"] = ce
        for seed in sens_seeds:
            ALL_ROWS += run_block_resumable(ds, SENS_SCEN, SENS_SCEN, seed, methods=SENS_METHODS, costs=costs, tag="sens_ce"); flush()
    for K in CONFIG["CAP_GRID"]:
        if K == CONFIG["CAPACITY"]: continue
        for seed in sens_seeds:
            ALL_ROWS += run_block_resumable(ds, SENS_SCEN, SENS_SCEN, seed, methods=SENS_METHODS, K=K, tag="sens_cap"); flush()

RES = flush()
RUNTIME_THIS_SESSION_MIN = (time.time() - T_START) / 60
_rt = OUT / "runtime_sessions.csv"      # wall-clock time is accumulated over resumable sessions
pd.DataFrame([dict(timestamp=time.strftime("%Y-%m-%d %H:%M:%S"), minutes=RUNTIME_THIS_SESSION_MIN)]).to_csv(_rt, mode="a", header=not _rt.exists(), index=False)
print(f"Session runtime: {RUNTIME_THIS_SESSION_MIN:.1f} min; total over sessions: {pd.read_csv(_rt).minutes.sum():.1f} min; rows = {len(RES)}")

# ---- integrity report of the missing-data design -------------------------------------------------
MASKS = pd.read_csv(MASK_LOG_CSV)
assert all(int(r.n_missing) == n_missing_target(round(r.target_rate * 100), r.n_cells) for r in MASKS.itertuples()), "a realised missing count deviates from round(rate*n*p)"
_mx = MASKS[MASKS.mechanism == "MIXED"]
assert ((_mx.n_mcar == _mx.n_missing // 3) & (_mx.n_mar == _mx.n_missing // 3) & (_mx.n_mcar + _mx.n_mar + _mx.n_mnar == _mx.n_missing)).all()
MASK_SUM = MASKS.groupby(["dataset", "scenario", "split"]).agg(target_rate=("target_rate", "first"), realised_min=("realised_rate", "min"),
            realised_max=("realised_rate", "max"), n_masks=("mask_seed", "nunique"), seeds=("mask_seed", lambda s: f"{s.min()}-{s.max()}")).round(6)
MASK_SUM.to_csv(OUT / "mask_rates_summary.csv"); print(MASK_SUM.to_string())

## 9. Summary tables, agreement of RMSE and regret, descriptive comparisons
Reporting unit = data set × setting (mechanism × rate). Seeds → within-setting variability. Settings of one data set share
the test fleet, so comparisons are descriptive; the blocked tests written to `blocked_comparisons_exploratory.csv` are not used in the paper.

In [ ]:
MAIN_METHODS = [m for m in CONFIG["METHODS"]] + ["oracle_perfect_info"]
INCOMPLETE_PRED = [m for m in ["xgb_native"] if m in CONFIG["METHODS"]]                      # prediction-only on incomplete data
UNCAL = [m for m in ["xgb_native", "oracle_complete_xgb"] if m in CONFIG["METHODS"]]           # prediction-only family (incl. complete-data reference)
CAL   = [m + "_margin" for m in UNCAL if m + "_margin" in CONFIG["METHODS"]]                   # same, regret-calibrated
DEC   = [m for m in ["xgb_quantile", "xgb_quantile_val", "dami"] if m in CONFIG["METHODS"]]    # cost-aware training
METRICS = ["rmse", "mae", "phm08", "failures", "early_pm", "downtime", "cost", "regret", "norm_regret"]

def boot_ci(x, n=2000, seed=0):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 2: return (np.nan, np.nan)
    rng = np.random.default_rng(seed)
    m = [rng.choice(x, len(x), replace=True).mean() for _ in range(n)]
    return (float(np.percentile(m, 2.5)), float(np.percentile(m, 97.5)))

main = RES[(RES.tag == "main") & RES.method.isin(MAIN_METHODS)].copy()
main["mechanism"] = main.scen_test.str.split("-").str[0]; main["rate"] = main.scen_test.str.split("-").str[1].astype(int)
summary = (main.groupby(["dataset", "scen_test", "method"])[METRICS].agg(["mean", "median", "std", "count"]).round(4))
summary.to_csv(OUT / "summary_main.csv")
print(summary.head(20).to_string())

In [ ]:
# --- RQ1: prediction error versus regret across the missingness settings -------------------------------------
# (a) native model: over the 20 mechanism x rate settings of a data set, do RMSE and regret move together?
blk = main.groupby(["dataset", "scen_test", "mechanism", "rate", "method"])[["rmse", "regret", "norm_regret", "cost"]].mean().reset_index()
rq1a = []
for ds, g in blk[blk.method == "xgb_native"].groupby("dataset"):
    rho, _ = stats.spearmanr(g["rmse"], g["regret"]) if len(g) >= 3 else (np.nan, np.nan)
    rq1a.append(dict(dataset=ds, n_settings=len(g), spearman_rmse_regret=rho,
                     rmse_min=g.rmse.min(), rmse_max=g.rmse.max(), rmse_range=g.rmse.max() - g.rmse.min(),
                     regret_min=g.regret.min(), regret_max=g.regret.max(), regret_range=g.regret.max() - g.regret.min(),
                     norm_regret_min=g.norm_regret.min(), norm_regret_max=g.norm_regret.max(),
                     best_rmse_setting=g.loc[g.rmse.idxmin(), "scen_test"], best_regret_setting=g.loc[g.regret.idxmin(), "scen_test"]))
RANK_SETTINGS = pd.DataFrame(rq1a); RANK_SETTINGS.to_csv(OUT / "rank_agreement_across_settings.csv", index=False)
print(RANK_SETTINGS.round(3).to_string())

# (b) within each data set x setting block: agreement of RMSE and regret rankings across the compared strategies
comp_methods = [m for m in CONFIG["METHODS"]]
rank_rows = []
for (ds, sc), g in blk[blk.method.isin(comp_methods)].groupby(["dataset", "scen_test"]):
    mm = g.set_index("method")
    if len(mm) < 3: continue
    rho, p = stats.spearmanr(mm["rmse"], mm["regret"])
    rank_rows.append(dict(dataset=ds, scenario=sc, n_methods=len(mm), spearman_rmse_regret=rho,
                          best_rmse=mm["rmse"].idxmin(), best_regret=mm["regret"].idxmin(),
                          disagree=int(mm["rmse"].idxmin() != mm["regret"].idxmin()),
                          rmse_range=float(mm["rmse"].max() - mm["rmse"].min()),
                          regret_range=float(mm["regret"].max() - mm["regret"].min())))
RANK = pd.DataFrame(rank_rows); RANK.to_csv(OUT / "rank_agreement.csv", index=False)
print(RANK.round(3).to_string())

# (c) mechanism and rate effects on the native model (means over seeds, then over rates / mechanisms)
MECH = blk[blk.method.isin(MAIN_METHODS)].groupby(["dataset", "method", "mechanism"])[["rmse", "norm_regret"]].mean().round(3)
RATE = blk[blk.method.isin(MAIN_METHODS)].groupby(["dataset", "method", "rate"])[["rmse", "norm_regret"]].mean().round(3)
MECH.to_csv(OUT / "by_mechanism.csv"); RATE.to_csv(OUT / "by_rate.csv")

In [ ]:
# --- RQ2: prediction-only vs regret-calibrated vs cost-aware training, per block ---------------------------------
cal_rows = []
for (ds, sc), g in main.groupby(["dataset", "scen_test"]):
    mm = g.groupby("method")[["rmse", "regret", "norm_regret", "failures", "early_pm", "margin"]].mean()
    get = lambda m, c="regret": float(mm.loc[m, c]) if m in mm.index else np.nan
    d = mm.loc[[m for m in DEC if m in mm.index]]
    cal_rows.append(dict(dataset=ds, scenario=sc, native_regret=get("xgb_native"), native_cal_regret=get("xgb_native_margin"),
        complete_regret=get("oracle_complete_xgb"), complete_cal_regret=get("oracle_complete_xgb_margin"),
        native_rmse=get("xgb_native", "rmse"), native_cal_rmse=get("xgb_native_margin", "rmse"),
        ratio_native_over_cal=get("xgb_native") / get("xgb_native_margin") if get("xgb_native_margin") else np.nan,
        best_dec=d.regret.idxmin() if len(d) else None, best_dec_regret=d.regret.min() if len(d) else np.nan,
        margin_native=get("xgb_native_margin", "margin")))
CALIB = pd.DataFrame(cal_rows); CALIB.to_csv(OUT / "calibration_analysis.csv", index=False)
print(CALIB.round(2).to_string())
# spread of regret across the 20 missingness settings before and after calibration (what is the missingness setting "worth"?)
_sp = []
for ds, g in CALIB.groupby("dataset"):
    oc = main[(main.dataset == ds) & (main.method == "oracle_perfect_info")].oracle_cost.mean()
    rho, _ = stats.spearmanr(g.native_regret, g.native_cal_regret) if len(g) >= 3 else (np.nan, np.nan)
    _sp.append(dict(dataset=ds, oracle_cost=oc, spread_before=g.native_regret.max() - g.native_regret.min(),
                    spread_after=g.native_cal_regret.max() - g.native_cal_regret.min(), spearman_before_after=rho))
CALSPREAD = pd.DataFrame(_sp); CALSPREAD["spread_before_frac"] = CALSPREAD.spread_before / CALSPREAD.oracle_cost
CALSPREAD["spread_after_frac"] = CALSPREAD.spread_after / CALSPREAD.oracle_cost
CALSPREAD.to_csv(OUT / "calibration_spread.csv", index=False); print(CALSPREAD.round(3).to_string())

In [ ]:
# --- Descriptive pairwise differences per data set (mean over settings; number of settings won by the reference) ----
PAIRS_DEF = [("xgb_native_margin", "xgb_native"), ("xgb_native_margin", "oracle_complete_xgb"), ("xgb_quantile_val", "xgb_native"),
             ("xgb_native_margin", "xgb_quantile_val"), ("xgb_native_margin", "dami"), ("dami", "xgb_native"),
             ("xgb_quantile", "xgb_native"), ("oracle_complete_xgb", "xgb_native")]
_b = main.groupby(["dataset", "scen_test", "method"])["norm_regret"].mean().unstack("method")
pr = []
for ref, comp in PAIRS_DEF:
    if ref not in _b or comp not in _b: continue
    for ds, g in (_b[comp] - _b[ref]).groupby(level="dataset"):
        pr.append(dict(reference=ref, comparator=comp, dataset=ds, mean_diff=float(g.mean()), ref_lower_in=int((g > 0).sum()), n_settings=int(g.notna().sum())))
PAIRS = pd.DataFrame(pr); PAIRS.to_csv(OUT / "pairwise_differences.csv", index=False); print(PAIRS.round(3).to_string())

# exploratory blocked tests (NOT used in the paper: settings of one data set share the fleet and are not independent)
def blocked_comparison(df, ref, metric="regret", methods=None):
    blocks = df.groupby(["dataset", "scen_test", "method"])[metric].mean().unstack("method")
    out = []
    for m in methods:
        if m not in blocks or ref not in blocks: continue
        d = (blocks[m] - blocks[ref]).dropna()
        if len(d) == 0: continue
        wins = int((d > 0).sum()); losses = int((d < 0).sum())
        try: p = float(stats.wilcoxon(d).pvalue) if len(d) >= 6 else np.nan
        except ValueError: p = 1.0
        out.append(dict(reference=ref, comparator=m, metric=metric, n_blocks=len(d), mean_diff=float(d.mean()), ref_wins=wins, ref_losses=losses, p_raw_exploratory=p))
    return pd.DataFrame(out)
CMP = pd.concat([blocked_comparison(main, r, met, [m for m in CONFIG["METHODS"] if m != r]) for r in ["xgb_native_margin", "dami"] for met in ["regret", "rmse"]], ignore_index=True)
CMP.to_csv(OUT / "blocked_comparisons_exploratory.csv", index=False)

In [ ]:
# --- RQ4: shift, and regret as a function of the missing rate within each mechanism -------------------------------
shift = RES[(RES.tag == "shift") & RES.method.isin(MAIN_METHODS)]
SHIFT_SUM = shift.groupby(["dataset", "scen_train", "scen_test", "method"])[["rmse", "cost", "regret", "norm_regret", "failures"]].agg(["mean", "std"]).round(3)
SHIFT_SUM.to_csv(OUT / "summary_shift.csv"); print(SHIFT_SUM.to_string())

slope_rows = []
for (ds, mech, m), g in main.groupby(["dataset", "mechanism", "method"]):
    gm = g.groupby("rate")["norm_regret"].mean()
    if len(gm) >= 2 and m != "oracle_perfect_info":
        slope_rows.append(dict(dataset=ds, mechanism=mech, method=m, norm_regret_slope_per_pct_missing=np.polyfit(gm.index.values, gm.values, 1)[0],
                               norm_regret_at_rates=gm.round(3).to_dict()))
SLOPE = pd.DataFrame(slope_rows); SLOPE.to_csv(OUT / "degradation_slope.csv", index=False); print(SLOPE.to_string())

In [ ]:
# --- sensitivity to cost structure and capacity ---------------------------------------------------------------
_sel = lambda tags: RES[(RES.tag.isin(tags)) & (RES.scen_test == SENS_SCEN) & (RES.scen_train == SENS_SCEN) & (RES.seed.isin(sens_seeds))]
_sm = SENS_METHODS + ["oracle_perfect_info"]
sens_ce = _sel(["sens_ce", "main"]); sens_ce = sens_ce[sens_ce.method.isin(_sm) & (sens_ce.c_f == CONFIG["COSTS"]["c_f"]) & (sens_ce.capacity == CONFIG["CAPACITY"])]
SENS_CE = sens_ce.groupby(["dataset", "c_e", "method"])[["rmse", "cost", "regret", "norm_regret", "failures", "early_pm", "margin"]].mean().round(3)
SENS_CE.to_csv(OUT / "sensitivity_early_cost.csv")
sens_cf = _sel(["sens_cf"]); sens_cf = sens_cf[sens_cf.method.isin(_sm)]
SENS_CF = sens_cf.groupby(["dataset", "c_f", "method"])[["rmse", "cost", "regret", "norm_regret", "failures", "margin"]].mean().round(3)
SENS_CF.to_csv(OUT / "sensitivity_cost_ratio.csv")
sens_cap = _sel(["sens_cap", "main"]); sens_cap = sens_cap[sens_cap.method.isin(_sm) & (sens_cap.c_f == CONFIG["COSTS"]["c_f"]) & (sens_cap.c_e == CONFIG["COSTS"]["c_e"])]
SENS_CAP = sens_cap.groupby(["dataset", "capacity", "method"])[["rmse", "cost", "oracle_cost", "regret", "norm_regret", "failures", "val_oracle_failures"]].mean().round(3)
SENS_CAP.to_csv(OUT / "sensitivity_capacity.csv"); print(SENS_CAP.to_string())

# lambda grid and selected hyper-parameters
lam_grid = RES[RES.tag.astype(str).str.contains("lambda_grid") & RES.tag.astype(str).str.startswith("main")].copy()
lam_grid["lamv"] = lam_grid.method.str.replace("dami_lam", "", regex=False).astype(float)
LAM = lam_grid.groupby(["dataset", "lamv"])[["rmse", "norm_regret", "regret", "failures"]].mean().round(3)
LAM.to_csv(OUT / "lambda_grid.csv"); print(LAM.to_string())
SELECTED = dict(
    margin=main[main.method == "xgb_native_margin"].groupby(["dataset", "mechanism"])["margin"].agg(["min", "max", "mean"]),
    alpha=main[main.method == "xgb_quantile_val"].groupby("dataset")["quantile_alpha"].value_counts().unstack(fill_value=0),
    lam=main[main.method == "dami"].groupby("dataset")["lam"].value_counts().unstack(fill_value=0))
for k_, v_ in SELECTED.items(): v_.to_csv(OUT / f"selected_{k_}.csv"); print(k_); print(v_.to_string())
# capacity pressure of validation vs. deployment fleets (is the validation fleet failure-free?)
REGIME = main[main.method == "oracle_perfect_info"].groupby("dataset").agg(test_oracle_cost=("oracle_cost", "mean"), test_oracle_failures=("failures", "mean"),
            val_oracle_failures_mean=("val_oracle_failures", "mean"), val_oracle_failures_max=("val_oracle_failures", "max"),
            n_val_units_min=("n_val_units", "min"), n_val_units_max=("n_val_units", "max"), n_test_units=("n_test_units", "first"),
            horizon_min=("horizon", "min"), horizon_max=("horizon", "max"))
REGIME.to_csv(OUT / "regime_check.csv"); print(REGIME.to_string())
HORIZON = RES.groupby(["dataset", "capacity"])["horizon"].agg(["min", "median", "max", "count"]); HORIZON.to_csv(OUT / "horizon_stats.csv")

In [ ]:

PAPER = OUT / "paper"; (PAPER / "tables").mkdir(parents=True, exist_ok=True); (PAPER / "figs").mkdir(parents=True, exist_ok=True)
NAMES = {"xgb_native": "Native GBR", "oracle_complete_xgb": "Complete-data GBR", "xgb_native_margin": "Native GBR, calibrated",
         "oracle_complete_xgb_margin": "Complete-data GBR, calibrated", "xgb_quantile": r"Quantile GBR ($\alpha$ theoretical)",
         "xgb_quantile_val": r"Quantile GBR ($\alpha$ validated)", "dami": r"Hybrid loss GBR ($\lambda$ validated)"}
SHORT = {"xgb_native": "Native", "xgb_native_margin": "Native (cal.)", "xgb_quantile": r"Quantile $\alpha_{th}$",
         "xgb_quantile_val": r"Quantile $\alpha_{val}$", "dami": "Hybrid loss", "oracle_complete_xgb": "Compl. GBR"}
FAMILIES = [("Prediction-only", UNCAL), ("Regret-calibrated", CAL), ("Cost-aware training", DEC)]
DS_ORDER = [d for d in ["FD001", "FD003", "FD002", "FD004"] if d in CONFIG["DATASETS"]]
WATERMARK = "% SYNTHETIC SMOKE TEST - DO NOT REPORT\n" if CONFIG["SMOKE_TEST"] else ""

def _write(name, colspec, header_lines, body_lines, width=r"\textwidth"):
    if width is None:   # natural width (the manuscript scales the table with \\resizebox)
        tex = WATERMARK + "\\begin{tabular}{@{}%s@{}}\\toprule\n" % colspec + "\n".join(header_lines) + " \\midrule\n" + "\n".join(body_lines) + "\n\\bottomrule\\end{tabular}\n"
    else:
        tex = WATERMARK + "\\begin{tabular*}{%s}{@{\\extracolsep{\\fill}}%s@{}}\\toprule\n" % (width, colspec) + "\n".join(header_lines) + " \\midrule\n" + "\n".join(body_lines) + "\n\\bottomrule\\end{tabular*}\n"
    (PAPER / "tables" / f"tab_{name}.tex").write_text(tex)
def _f(x, nd=2): return "--" if x is None or (isinstance(x, float) and np.isnan(x)) else f"{x:.{nd}f}"
def _pm(m, s, nd=2): return "--" if np.isnan(m) else f"{_f(m, nd)}$\\pm${_f(s, nd)}"
def _family_rows(fn, ncols):
    out = []
    for fam, ms in FAMILIES:
        out.append("\\multicolumn{%d}{@{}l}{\\emph{%s}} \\\\" % (ncols, fam))
        out += [NAMES[m] + " & " + " & ".join(fn(m)) + " \\\\" for m in ms]
    return out

bm = main.groupby(["dataset", "scen_test", "method"])[METRICS + ["oracle_cost"]].mean()      # block means over seeds
bs_ = main.groupby(["dataset", "scen_test", "method"])[METRICS].std()                          # seed-to-seed s.d. per block

# ---- overview: normalised regret (mean +- s.d. across the 20 settings) and RMSE, per data set -------------------
def _ov(m):
    cells = []
    for ds in DS_ORDER:
        try: v = bm.xs((ds, m), level=("dataset", "method"))
        except KeyError: cells += ["--", "--"]; continue
        cells += [_pm(v.norm_regret.mean(), v.norm_regret.std()), _f(v.rmse.mean(), 1)]
    return cells
_write("overview", "l" + "cc" * len(DS_ORDER),
       [" & " + " & ".join("\\multicolumn{2}{c}{%s}" % d for d in DS_ORDER) + " \\\\", "Strategy & " + " & ".join(["Regret & RMSE"] * len(DS_ORDER)) + " \\\\"],
       _family_rows(_ov, 1 + 2 * len(DS_ORDER)) + ["\\midrule Perfect-information plan & " + " & ".join(["0 & --"] * len(DS_ORDER)) + " \\\\"])

# ---- per data set: normalised regret by setting (mean +- s.d. over seeds) ------------------------------------------
for ds in DS_ORDER:
    def _sc(m, ds=ds):
        return [_pm(bm.norm_regret.get((ds, sc, m), np.nan), bs_.norm_regret.get((ds, sc, m), np.nan)) for sc in CONFIG["SCENARIOS"]]
    _write(f"byscenario_{ds}", "l" + "c" * len(CONFIG["SCENARIOS"]), ["Strategy & " + " & ".join(CONFIG["SCENARIOS"]) + " \\\\"],
           _family_rows(_sc, 1 + len(CONFIG["SCENARIOS"])), width=None)
    # prediction and operational metrics averaged over the settings
    def _op(m, ds=ds):
        try: v = bm.xs((ds, m), level=("dataset", "method"))
        except KeyError: return ["--"] * 6
        return [_pm(v.rmse.mean(), v.rmse.std(), 1), _f(v.mae.mean(), 1), _f(v.phm08.mean(), 0), _f(v.failures.mean(), 1), _f(v.early_pm.mean(), 1), _f(v.downtime.mean(), 0)]
    _o = bm.xs((ds, "oracle_perfect_info"), level=("dataset", "method"))
    _write(f"ops_{ds}", "lcccccc", ["Strategy & RMSE & MAE & PHM08 & Failures & Early PM & Downtime \\\\"],
           _family_rows(_op, 7) + ["\\midrule Perfect-information plan & 0 & 0 & 0 & %s & %s & %s \\\\" % (_f(_o.failures.mean(), 1), _f(_o.early_pm.mean(), 0), _f(_o.downtime.mean(), 0))])

# ---- by mechanism (mean over rates) and by rate (mean over mechanisms): native vs calibrated vs validated quantile ---
_mm = [m for m in ["xgb_native", "xgb_native_margin", "xgb_quantile", "xgb_quantile_val", "dami"] if m in CONFIG["METHODS"]]
_g = main.groupby(["dataset", "mechanism", "rate", "method"])[["rmse", "norm_regret"]].mean().reset_index()
rows_ = []
for ds in DS_ORDER:
    for i, mech in enumerate(CONFIG["MECHANISMS"]):
        s = _g[(_g.dataset == ds) & (_g.mechanism == mech)]
        rows_.append((ds if i == 0 else "") + f" & {mech} & " + _f(s[s.method == "xgb_native"].rmse.mean(), 1) + " & " + " & ".join(_f(s[s.method == m].norm_regret.mean()) for m in _mm) + " \\\\" + ("\n\\midrule" if i == len(CONFIG["MECHANISMS"]) - 1 and ds != DS_ORDER[-1] else ""))
_write("mechanism", "llc" + "c" * len(_mm), ["Data set & Mechanism & RMSE (native) & " + " & ".join(SHORT[m] for m in _mm) + " \\\\"], rows_)
rows_ = []
for ds in DS_ORDER:
    for i, r in enumerate(CONFIG["RATES"]):
        s = _g[(_g.dataset == ds) & (_g.rate == r)]
        rows_.append((ds if i == 0 else "") + f" & {r}\\% & " + _f(s[s.method == "xgb_native"].rmse.mean(), 1) + " & " + " & ".join(_f(s[s.method == m].norm_regret.mean()) for m in _mm) + " \\\\" + ("\n\\midrule" if i == len(CONFIG["RATES"]) - 1 and ds != DS_ORDER[-1] else ""))
_write("rate", "llc" + "c" * len(_mm), ["Data set & Rate & RMSE (native) & " + " & ".join(SHORT[m] for m in _mm) + " \\\\"], rows_)

# ---- RQ1 agreement table -------------------------------------------------------------------------------------
rows_ = []
for ds in DS_ORDER:
    a = RANK_SETTINGS[RANK_SETTINGS.dataset == ds]; b = RANK[RANK.dataset == ds]
    if len(a) == 0: continue
    a = a.iloc[0]
    rows_.append(f"{ds} & {_f(a.spearman_rmse_regret)} & {_f(a.rmse_range, 1)} / {_f(a.regret_range, 0)} & {_f(b.spearman_rmse_regret.mean())} & {int((b.spearman_rmse_regret < 0).sum())}/{len(b)} & {int(b.disagree.sum())}/{len(b)} \\\\")
rows_.append(f"\\midrule All & {_f(RANK_SETTINGS.spearman_rmse_regret.mean())} & {_f(RANK_SETTINGS.rmse_range.mean(), 1)} / {_f(RANK_SETTINGS.regret_range.mean(), 0)} & {_f(RANK.spearman_rmse_regret.mean())} & {int((RANK.spearman_rmse_regret < 0).sum())}/{len(RANK)} & {int(RANK.disagree.sum())}/{len(RANK)} \\\\")
_write("rank", "lccccc", ["Data set & $\\rho_S$ across settings & RMSE / regret range & Mean $\\rho_S$ across strategies & Settings with $\\rho_S<0$ & Disagree \\\\"], rows_)

# ---- calibration spread, pairwise differences -----------------------------------------------------------------
_write("calspread", "lccc", ["Data set & Spread before calibration & Spread after calibration & $\\rho_S$(before, after) \\\\"],
       [f"{r.dataset} & {_f(r.spread_before, 1)} ({_f(r.spread_before_frac)}) & {_f(r.spread_after, 1)} ({_f(r.spread_after_frac)}) & {_f(r.spearman_before_after)} \\\\"
        for r in CALSPREAD.set_index("dataset").loc[[d for d in DS_ORDER if d in set(CALSPREAD.dataset)]].reset_index().itertuples()])
rows_ = []
for ref, comp in PAIRS_DEF:
    s = PAIRS[(PAIRS.reference == ref) & (PAIRS.comparator == comp)].set_index("dataset")
    if len(s) == 0: continue
    rows_.append(f"{SHORT[ref]} & {SHORT[comp]} & " + " & ".join((f"{s.loc[d, 'mean_diff']:+.2f} ({s.loc[d, 'ref_lower_in']}/{s.loc[d, 'n_settings']})" if d in s.index else "--") for d in DS_ORDER) + " \\\\")
_write("pairs", "ll" + "c" * len(DS_ORDER), ["Reference & Comparator & " + " & ".join(DS_ORDER) + " \\\\"], rows_)

# ---- sensitivity tables ------------------------------------------------------------------------------------
def _sens(tab, level, name, extra_cols=None, fmt=lambda v: f"{v:g}"):
    rows_ = []
    for ds in DS_ORDER:
        if ds not in tab.index.get_level_values("dataset"): continue
        vals = sorted(tab.xs(ds, level="dataset").index.get_level_values(level).unique())
        for i, v in enumerate(vals):
            s = tab.xs((ds, v), level=("dataset", level))
            cells = [_f(s.norm_regret.get(m, np.nan)) for m in SENS_METHODS]
            if extra_cols: cells += [_f(s.loc["oracle_perfect_info", c], nd) if "oracle_perfect_info" in s.index else "--" for c, nd in extra_cols]
            rows_.append((ds if i == 0 else "") + f" & {fmt(v)} & " + " & ".join(cells) + " \\\\" + ("\n\\midrule" if i == len(vals) - 1 and ds != DS_ORDER[-1] else ""))
    return rows_
_hdr = lambda first, extra="": [f"Data set & {first} & " + " & ".join(SHORT[m] for m in SENS_METHODS) + extra + " \\\\"]
_write("cap", "lc" + "c" * (len(SENS_METHODS) + 3), _hdr("$K$", " & Oracle cost & Oracle fail. & Val. oracle fail."),
       _sens(SENS_CAP, "capacity", "cap", extra_cols=[("oracle_cost", 2), ("failures", 1), ("val_oracle_failures", 1)]))
_write("cf", "lc" + "c" * len(SENS_METHODS), _hdr("$c_{\\mathrm{F}}/c_{\\mathrm{PM}}$"), _sens(SENS_CF, "c_f", "cf"))
_write("ce", "lc" + "c" * len(SENS_METHODS), _hdr("$c_{\\mathrm{E}}$"), _sens(SENS_CE, "c_e", "ce"))

# ---- shift table: shifted value (matched-setting value of the test setting in brackets) ------------------------------
if len(shift):
    sh = shift.groupby(["dataset", "scen_train", "scen_test", "method"])["norm_regret"].mean()
    pairs_ = sorted(set((a, b) for (_, a, b, _) in sh.index))
    rows_ = []
    for fam, ms in FAMILIES:
        for m in ms:
            cells = []
            for ds in DS_ORDER:
                for a, b in pairs_:
                    v = sh.get((ds, a, b, m), np.nan); ref = bm.norm_regret.get((ds, b, m), np.nan)
                    cells.append(f"{_f(v)} ({_f(ref)})")
            rows_.append(NAMES[m] + " & " + " & ".join(cells) + " \\\\")
    _write("shift", "l" + "c" * (len(DS_ORDER) * len(pairs_)),
           [" & " + " & ".join("\\multicolumn{%d}{c}{%s}" % (len(pairs_), d) for d in DS_ORDER) + " \\\\", "Strategy & " + " & ".join(f"{a}$\\to${b}" for _ in DS_ORDER for a, b in pairs_) + " \\\\"], rows_)

# ---- lambda grid, selected values, mask rates, regime check ----------------------------------------------------------
rows_ = [f"{l:g} & " + " & ".join((f"{_f(LAM.norm_regret.get((d, l), np.nan))} / {_f(LAM.rmse.get((d, l), np.nan), 1)}") for d in DS_ORDER) + " \\\\" for l in CONFIG["LAMBDA_GRID"]]
_d = bm.xs("dami", level="method") if "dami" in CONFIG["METHODS"] else None
if _d is not None: rows_.append("\\midrule Validation-selected & " + " & ".join(f"{_f(_d.xs(d, level='dataset').norm_regret.mean())} / {_f(_d.xs(d, level='dataset').rmse.mean(), 1)}" for d in DS_ORDER) + " \\\\")
_write("lambda", "l" + "c" * len(DS_ORDER), ["$\\lambda$ & " + " & ".join(DS_ORDER) + " \\\\"], rows_, width=r"\columnwidth")
_dag = lambda v, grid: f"{v:g}" + ("$^\\dagger$" if v in (min(grid), max(grid)) else "")
rows_ = []
for ds in DS_ORDER:
    mg = main[(main.dataset == ds) & (main.method == "xgb_native_margin")].groupby("mechanism")["margin"].agg(["min", "max"])
    al = main[(main.dataset == ds) & (main.method == "xgb_quantile_val")]["quantile_alpha"].value_counts().sort_index()
    lm = main[(main.dataset == ds) & (main.method == "dami")]["lam"].value_counts().sort_index()
    rows_.append(f"{ds} & " + " & ".join((f"{mg.loc[me, 'min']:g}--{mg.loc[me, 'max']:g}" if me in mg.index else "--") for me in CONFIG["MECHANISMS"]) + " & "
                 + ", ".join(f"{_dag(a, CONFIG['QUANTILE_GRID'])}: {n}" for a, n in al.items()) + " & " + ", ".join(f"{_dag(l, [x for x in CONFIG['LAMBDA_GRID'] if x > 0])}: {n}" for l, n in lm.items()) + " \\\\")
_write("selection", "l" + "c" * (len(CONFIG["MECHANISMS"]) + 2), ["Data set & " + " & ".join(f"$\\delta$: {me}" for me in CONFIG["MECHANISMS"]) + " & $\\alpha$ selected & $\\lambda$ selected \\\\"], rows_, width=None)
_ms = MASKS.groupby(["scenario", "split"]).agg(t=("target_rate", "first"), lo=("realised_rate", "min"), hi=("realised_rate", "max"), n=("mask_file", "nunique"),
                                              c1=("n_mcar", "sum"), c2=("n_mar", "sum"), c3=("n_mnar", "sum"), k=("n_missing", "sum"))
rows_ = []
for sc in CONFIG["SCENARIOS"]:
    for i, sp in enumerate(["train", "val", "test"]):
        if (sc, sp) not in _ms.index: continue
        r = _ms.loc[(sc, sp)]
        rows_.append((sc if i == 0 else "") + f" & {sp} & {r.t:.2f} & {r.lo:.4f} & {r.hi:.4f} & {int(r.n)} & {r.c1 / r.k:.3f} / {r.c2 / r.k:.3f} / {r.c3 / r.k:.3f} \\\\")
_write("maskrates", "llccccc", ["Setting & Split & Target rate & Realised (min) & Realised (max) & Masks & Share MCAR / MAR / MNAR \\\\"], rows_)
_write("regime", "lccccccc", ["Data set & Test units & Oracle cost & Oracle failures & Val. units & Val. oracle failures (mean) & (max) & Horizon \\\\"],
       [f"{d} & {int(REGIME.loc[d, 'n_test_units'])} & {_f(REGIME.loc[d, 'test_oracle_cost'])} & {_f(REGIME.loc[d, 'test_oracle_failures'], 1)} & {int(REGIME.loc[d, 'n_val_units_min'])}--{int(REGIME.loc[d, 'n_val_units_max'])} & {_f(REGIME.loc[d, 'val_oracle_failures_mean'], 2)} & {_f(REGIME.loc[d, 'val_oracle_failures_max'], 0)} & {int(REGIME.loc[d, 'horizon_min'])}--{int(REGIME.loc[d, 'horizon_max'])} \\\\" for d in DS_ORDER if d in REGIME.index])

# =============================== figures ===============================
FAM_STYLE = {"Prediction-only": ("o", "tab:blue"), "Regret-calibrated": ("s", "tab:red"), "Cost-aware training": ("^", "tab:green")}
def _grid(n): return plt.subplots(2, 2, figsize=(9, 6.5)) if n > 2 else plt.subplots(1, max(n, 1), figsize=(4.5 * max(n, 1), 3.6), squeeze=False)
def _mark(fig):
    if CONFIG["SMOKE_TEST"]: fig.text(0.5, 0.5, "SYNTHETIC SMOKE TEST", fontsize=30, color="red", alpha=.3, ha="center", va="center", rotation=30)

fig, axes = _grid(len(DS_ORDER))
for ax, ds in zip(np.ravel(axes), DS_ORDER):
    for fam, ms in FAMILIES:
        mk, col = FAM_STYLE[fam]
        for m in ms:
            try: v = bm.xs((ds, m), level=("dataset", "method"))
            except KeyError: continue
            ax.errorbar(v.rmse.mean(), v.norm_regret.mean(), xerr=v.rmse.std(), yerr=v.norm_regret.std(), fmt=mk, color=col, capsize=2,
                        mfc="white" if m.startswith("oracle_complete") else col, label=fam if m == ms[0] else None)
            ax.annotate(SHORT.get(m, NAMES[m]).replace("$", "").replace("\\alpha_{th}", "a_th").replace("\\alpha_{val}", "a_val"), (v.rmse.mean(), v.norm_regret.mean()), fontsize=6, xytext=(3, 3), textcoords="offset points")
    ax.set_title(ds); ax.set_xlabel("RUL RMSE (cycles)"); ax.set_ylabel("Normalized regret"); ax.grid(alpha=.3)
np.ravel(axes)[0].legend(fontsize=7); _mark(fig); fig.tight_layout(); fig.savefig(PAPER / "figs" / "pareto_datasets.pdf"); plt.close(fig)

fig, axes = _grid(len(DS_ORDER)); xs = np.arange(len(CONFIG["SCENARIOS"]))
for ax, ds in zip(np.ravel(axes), DS_ORDER):
    for fam, ms in FAMILIES:
        mk, col = FAM_STYLE[fam]
        A = np.array([[bm.norm_regret.get((ds, sc, m), np.nan) for sc in CONFIG["SCENARIOS"]] for m in ms if not m.startswith("oracle_complete")], float)
        if A.size == 0: continue
        ax.fill_between(xs, np.nanmin(A, 0), np.nanmax(A, 0), color=col, alpha=.25); ax.plot(xs, np.nanmedian(A, 0), color=col, marker=mk, ms=3, label=fam)
    ax.set_xticks(xs); ax.set_xticklabels(CONFIG["SCENARIOS"], rotation=60, fontsize=6); ax.set_title(ds); ax.set_ylabel("Normalized regret"); ax.grid(alpha=.3)
np.ravel(axes)[0].legend(fontsize=7); _mark(fig); fig.tight_layout(); fig.savefig(PAPER / "figs" / "family_by_scenario.pdf"); plt.close(fig)

fig, axes = _grid(len(DS_ORDER))
for ax, ds in zip(np.ravel(axes), DS_ORDER):
    if ds not in SENS_CAP.index.get_level_values("dataset"): continue
    t = SENS_CAP.xs(ds, level="dataset")
    for m in SENS_METHODS:
        if m in t.index.get_level_values("method"):
            s = t.xs(m, level="method"); ax.plot(s.index, s.norm_regret, marker="o", ms=3, label=SHORT[m].replace("$", ""))
    if "oracle_perfect_info" in t.index.get_level_values("method"):
        o = t.xs("oracle_perfect_info", level="method")
        for K_ in o.index[o.failures > 0]: ax.axvspan(K_ - .4, K_ + .4, color="grey", alpha=.2)
    ax.set_title(ds); ax.set_xlabel("Capacity K (units per period)"); ax.set_ylabel("Normalized regret"); ax.set_xticks(CONFIG["CAP_GRID"]); ax.grid(alpha=.3)
np.ravel(axes)[0].legend(fontsize=6); _mark(fig); fig.tight_layout(); fig.savefig(PAPER / "figs" / "capacity.pdf"); plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6))
for ax, tab, lev, lab in [(axes[0], SENS_CE, "c_e", "(a) early-replacement cost $c_E$"), (axes[1], SENS_CF, "c_f", "(b) failure-cost ratio $c_F/c_{PM}$")]:
    for ds, ls in [(d, l) for d, l in [("FD001", "-"), ("FD002", "--")] if d in tab.index.get_level_values("dataset")]:
        t = tab.xs(ds, level="dataset")
        for i, m in enumerate(SENS_METHODS):
            if m in t.index.get_level_values("method"):
                s = t.xs(m, level="method"); ax.plot(s.index, s.norm_regret, ls=ls, marker="o", ms=3, color=f"C{i}", label=f"{SHORT[m]} ({ds})".replace("$", ""))
    ax.set_xlabel(lab); ax.set_ylabel("Normalized regret"); ax.grid(alpha=.3)
axes[0].legend(fontsize=5, ncol=2); _mark(fig); fig.tight_layout(); fig.savefig(PAPER / "figs" / "costs.pdf"); plt.close(fig)

fig, ax = plt.subplots(figsize=(4.5, 3.4))
for ds in DS_ORDER:
    if ds in LAM.index.get_level_values("dataset"):
        s = LAM.xs(ds, level="dataset"); ax.plot(s.rmse, s.norm_regret, marker="o", ms=3, label=ds)
ax.set_xlabel("RUL RMSE (cycles)"); ax.set_ylabel("Normalized regret"); ax.legend(fontsize=7); ax.grid(alpha=.3)
_mark(fig); fig.tight_layout(); fig.savefig(PAPER / "figs" / "lambda.pdf"); plt.close(fig)
print("paper artefacts written to", PAPER)

## 11. Compact summary


In [ ]:
def _clean(o):
    if isinstance(o, dict): return {str(k): _clean(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_clean(v) for v in o]
    if isinstance(o, (np.floating, float)): return None if np.isnan(o) else round(float(o), 4)
    if isinstance(o, (np.integer,)): return int(o)
    return o

summary_json = dict(
    env=ENV, config={k: v for k, v in CONFIG.items() if k not in ("XGB_PARAMS",)}, xgb_params=CONFIG["XGB_PARAMS"],
    SMOKE_TEST_DO_NOT_REPORT=CONFIG["SMOKE_TEST"],
    n_blocks=dict(main=int(main.groupby(["dataset", "scen_test", "seed"]).ngroups), shift=int(shift.groupby(["dataset", "scen_train", "scen_test", "seed"]).ngroups),
                  evaluations=int(len(RES))),
    mask_rates=_clean(MASK_SUM.reset_index().to_dict("records")),
    regime_check=_clean(REGIME.reset_index().to_dict("records")),
    horizon=_clean(HORIZON.reset_index().to_dict("records")),
    main_mean_over_seeds=_clean(main.groupby(["dataset", "scen_test", "method"])[["rmse", "mae", "failures", "early_pm", "downtime", "cost", "oracle_cost", "regret", "norm_regret", "margin", "lam", "quantile_alpha"]].mean().round(3).reset_index().to_dict("records")),
    main_std_over_seeds=_clean(main.groupby(["dataset", "scen_test", "method"])[["rmse", "regret", "norm_regret"]].std().round(3).reset_index().to_dict("records")),
    rank_agreement_across_settings=_clean(RANK_SETTINGS.to_dict("records")),
    rank_agreement_across_strategies=_clean(RANK.to_dict("records")),
    calibration_analysis=_clean(CALIB.to_dict("records")), calibration_spread=_clean(CALSPREAD.to_dict("records")),
    pairwise_differences=_clean(PAIRS.to_dict("records")),
    by_mechanism=_clean(MECH.reset_index().to_dict("records")), by_rate=_clean(RATE.reset_index().to_dict("records")),
    shift=_clean(shift.groupby(["dataset", "scen_train", "scen_test", "method"])[["rmse", "cost", "regret", "norm_regret", "failures"]].mean().round(3).reset_index().to_dict("records")),
    degradation_slope=_clean(SLOPE.drop(columns=["norm_regret_at_rates"]).to_dict("records")) if len(SLOPE) else [],
    sensitivity_early_cost=_clean(SENS_CE.reset_index().to_dict("records")),
    sensitivity_cost_ratio=_clean(SENS_CF.reset_index().to_dict("records")),
    sensitivity_capacity=_clean(SENS_CAP.reset_index().to_dict("records")),
    lambda_grid=_clean(LAM.reset_index().to_dict("records")),
    selected={k: _clean(v.reset_index().to_dict("records")) for k, v in SELECTED.items()},
    runtime_total_min=round(float(pd.read_csv(OUT / "runtime_sessions.csv").minutes.sum()), 1),
)
(OUT / "SUMMARY_FOR_PAPER.json").write_text(json.dumps(summary_json, indent=1))
print(json.dumps(summary_json, indent=1)[:20000])
print("\n... full summary saved to", OUT / "SUMMARY_FOR_PAPER.json")

In [ ]:
# Zip all outputs for download (Colab) ------------------------------------------------------
import shutil
zip_path = shutil.make_archive("outputs", "zip", OUT)
print("Outputs archived:", zip_path)
if IN_COLAB:
    from google.colab import files
    files.download(zip_path)